# Chapter 4: Intermediate SQL


## Core Question

How can SQL express relationships clearly, retain unmatched rows when required, define a
reusable view, group several changes into one unit, and reject invalid data?


## Connection to Chapter 3

Chapter 3 used comma-separated inputs and a `WHERE` matching predicate. This chapter uses
explicit joins:

```sql
FROM student AS s
JOIN enrollment AS e ON e.student_id = s.student_id
```

For an inner join, a predicate in `ON` or `WHERE` may produce the same rows. For an outer
join, moving a predicate can change which unmatched rows survive.


## Teaching Summary

| Topic | Worked example and practice | Evidence to retain |
|---|---|---|
| Explicit and outer joins | Course enrollment with matched and unmatched rows | Join predicate and checked result |
| Views | Live department-query view and attempted update | Base-query comparison and error evidence |
| Transactions | Multi-statement change followed by rollback | Before, during, and after states |
| Constraints and reference actions | Feedback rules and deletion behavior | Passing and failing test cases |


## Prerequisites and Setup

You should be able to use aliases, `SELECT`, `WHERE`, grouping, `NULL`, and basic data
modification, and identify primary and foreign keys.

With the student package, run this command from the package root:

```powershell
Run the executable SQL cells below.
```

If separate files are provided, create a new SQLite database, run the Chapter 2 setup,
and then run the executable SQL lab cell. The materials were verified with SQLite 3.45.3.


## Learning Objectives

After completing this chapter, you should be able to:

1. Write explicit joins and explain every matching condition.
2. Compare `ON`, `USING`, and the risks of `NATURAL JOIN`.
3. Predict which rows an inner or left outer join retains.
4. Explain why a right-side condition in `ON` can differ from the same condition in
   `WHERE` after a left join.
5. Create and query a view and distinguish it from a stored result.
6. Use `COMMIT` and `ROLLBACK` to define a basic atomic unit of work.
7. Use `NOT NULL`, `UNIQUE`, `CHECK`, and foreign keys to express data rules.
8. Explain the effect of reject, `CASCADE`, and `SET NULL` reference actions.

Right and full outer joins, detailed view-update rules, materialized views, deferred
constraints, assertions, and authorization are extensions.


## 1. Explicit Inner Joins

An inner join retains row pairs that satisfy its join condition. Use `ON` for the
relationship between inputs and `WHERE` for additional filtering.

```sql
SELECT s.student_id, s.student_name,
       c.course_id, c.title, e.grade
FROM student AS s
JOIN enrollment AS e ON e.student_id = s.student_id
JOIN course AS c ON c.course_id = e.course_id
ORDER BY s.student_id, c.course_id;
```

The first join connects each enrollment to one student. The second connects it to one
course. The result has six rows. S101 taking FT210 remains valid even though the student
and course departments differ; department equality is not the enrollment relationship.

### Practice

List each enrollment's student email and course department name. Write the required
relations and both matching conditions before writing the complete query.

### Check Criteria

The result has six rows and retains S101/FT210 with the course department Finance. Too
many rows usually indicate a missing condition; too few may indicate an extra condition
that was not part of the requirement.


## 2. `USING` and `NATURAL JOIN`

`USING(column)` explicitly selects a same-named matching column. `NATURAL JOIN` uses all
same-named columns automatically, so a schema change or an unrelated name collision can
silently change the result.

```sql
SELECT c.course_id, c.title, d.dept_name
FROM course AS c
JOIN department AS d USING (dept_code)
ORDER BY c.course_id;
```

The result has one department for each of the four courses.

### Counterexample

```sql
SELECT student_id, student_name, course_id, title
FROM student
NATURAL JOIN enrollment
NATURAL JOIN course;
```

After the first join, both `course_id` and `dept_code` are same-named columns for the next
natural join. The query incorrectly removes S101/FT210 because the student's department
does not equal the course's department. Explicit `ON` conditions return the correct six
rows.

Practice: rewrite the `USING` example with `ON c.dept_code = d.dept_code` and compare
rows and output columns.


## 3. Inner and Outer Joins

| Join type | Unmatched rows retained |
|---|---|
| `INNER JOIN` | None |
| `LEFT JOIN` | Left input |
| `RIGHT JOIN` | Right input |
| `FULL JOIN` | Both inputs |

The classroom core uses left outer join. Right and full joins are included only for
recognition and portability awareness.

### Worked Example

The lab temporarily adds course IS250 with no enrollment:

```sql
SELECT c.course_id, c.title, COUNT(e.student_id) AS enrollment_count
FROM course AS c
LEFT JOIN enrollment AS e ON e.course_id = c.course_id
GROUP BY c.course_id, c.title
ORDER BY c.course_id;
```

IS250 remains with count 0. Use `COUNT(e.student_id)`, not `COUNT(*)`, because the latter
would count the null-padded outer-join row as one.

### Practice

List every student and enrollment count, including a temporary student with no
enrollment. The temporary student must appear once with count 0.


## 4. Conditions in `ON` and `WHERE`

For a left join, `ON` first determines matches and then preserves unmatched left rows.
`WHERE` filters the completed join result.

```sql
SELECT c.course_id, e.student_id, e.grade
FROM course AS c
LEFT JOIN enrollment AS e
  ON e.course_id = c.course_id
 AND e.grade IN ('A', 'A-');
```

This retains all courses and attaches only high-grade enrollments. If the grade condition
is moved to `WHERE`, an unmatched course has `NULL` grade, the predicate becomes
`UNKNOWN`, and that course is removed.

### Predict and Check

- Requirement A: retain all courses and attach only passing enrollments.
- Requirement B: list only courses with at least one passing enrollment.

Place the grade condition for each requirement and explain whether an unmatched course
survives. The answer must use outer-join and `NULL` reasoning, not a memorized preference
for `ON`.


## 5. Views

A view is a virtual relation defined by a query. A regular view stores its definition and
is evaluated from current base data; it is not a fixed copy of the creation-time result.

```sql
CREATE VIEW course_enrollment_summary AS
SELECT c.course_id,
       c.title,
       COUNT(e.student_id) AS enrollment_count
FROM course AS c
LEFT JOIN enrollment AS e ON e.course_id = c.course_id
GROUP BY c.course_id, c.title;
```

```sql
SELECT course_id, enrollment_count
FROM course_enrollment_summary
WHERE enrollment_count >= 2;
```

The current result includes DB201 and FT210. A temporary base-table change alters the
view result; rollback restores it.

### Practice

Create `im_course(course_id, title, credits)` for IM courses and query its three-credit
rows. A temporary IM course added to the base table should appear through the view and
disappear after rollback.

### Extension: View Modification

A join or aggregate view may not map an update to one unambiguous base row. SQLite views
are read-only unless an `INSTEAD OF` trigger is supplied, while other DBMS products may
allow selected simple-view updates. Treat this as a product-specific rule.


## 6. Transaction Boundaries

A transaction groups statements into one unit of work. `COMMIT` keeps the transaction's
changes. `ROLLBACK` removes uncommitted changes. Tool-specific autocommit settings must be
checked explicitly.

```sql
BEGIN;

DELETE FROM enrollment
WHERE student_id = 'S101'
  AND course_id = 'FT210'
  AND term = '115-1';

INSERT INTO enrollment (student_id, course_id, term, grade)
VALUES ('S101', 'ML230', '115-1', NULL);

COMMIT;
```

If the insertion fails after an independently committed deletion, the course swap is
only half complete. One transaction allows the application to roll back the whole unit.

Practice: perform a different valid course swap inside a savepoint. Retain the before,
modified, and post-rollback rows.


## 7. Integrity Constraints

```sql
CREATE TABLE waitlist_entry (
    request_id INTEGER PRIMARY KEY,
    student_id TEXT NOT NULL,
    course_id TEXT NOT NULL,
    term TEXT NOT NULL DEFAULT '115-1',
    priority INTEGER NOT NULL CHECK (priority BETWEEN 1 AND 5),
    UNIQUE (student_id, course_id, term)
);
```

`UNIQUE` prevents duplicate requests for the same student, course, and term. `CHECK`
limits priority. `DEFAULT` supplies a value only when the insertion omits the column.

A `CHECK` constraint rejects `FALSE`, but a comparison with `NULL` may be `UNKNOWN`.
Therefore `CHECK (priority > 0)` does not replace `NOT NULL`.

### Practice

Predict which rule rejects a duplicate request, priority 8, and a `NULL` student
identifier. Test each failure separately; do not rely on the order in which a DBMS reports
multiple violations.


## 8. Foreign-Key Actions

```sql
FOREIGN KEY (student_id) REFERENCES student (student_id)
    ON DELETE CASCADE,
FOREIGN KEY (course_id) REFERENCES course (course_id)
```

- Reject/no action prevents deletion while dependent rows remain.
- `CASCADE` removes dependent rows.
- `SET NULL` writes `NULL`, so the referencing column must allow it.

The choice follows the data-life-cycle rule. Cascading a waitlist row after a student
leaves may be appropriate. Cascading historical enrollment rows may destroy records that
must be retained.

### Design Practice

Design `course_feedback` so rating is 1-5, each row references one existing composite
Enrollment key, and each enrollment has at most one feedback row. Test a valid row, a
rating of 6, and a reference to a missing enrollment.


## Common Errors

1. Using `NATURAL JOIN` without listing every same-named column.
2. Moving a right-side left-join condition to `WHERE` and losing unmatched rows.
3. Counting null-padded rows with `COUNT(*)`.
4. Treating a regular view as a stored snapshot.
5. Assuming that every view is updatable.
6. Depending on an unknown autocommit setting.
7. Using `CHECK` alone to reject `NULL`.
8. Selecting `CASCADE` without considering the data life cycle.


## Classroom and Individual Evidence

Review an AI-generated query containing a natural join, a left-join filter in `WHERE`, a
questionable view update, and an unexplained cascade. Judge its explicit relationship,
unmatched rows, base-row mapping, transaction boundary, and business rule.

Retain initial predictions, lab output, comparison reasons, and the individually corrected
version. Peer ranking does not directly determine the course grade.


## Chapter Summary

Explicit joins expose matching rules. Outer joins preserve selected unmatched rows, so
`ON` and `WHERE` are not interchangeable. A regular view stores a query definition.
Transactions define all-or-nothing work, and constraints make checkable data rules part of
the schema. Chapter 5 introduces selected advanced SQL features.


## After-Class Continuation

Revise one outer-join query so that its unmatched rows have a stated purpose, then design
one constraint test that should succeed and one that should fail. Retain the prediction,
observed result, and business rule represented by each test.


## Build and Inspect the Chapter Database

The executable cells use SQLite through Python's standard `sqlite3` module. Follow the
cells in order:

1. Open a database connection and enable foreign-key enforcement.
2. Execute the chapter's `CREATE TABLE` statements before inserting rows.
3. Load the synthetic example data.
4. Inspect the resulting tables, columns, primary keys, and foreign keys.
5. Check referential integrity before running the chapter queries.
6. Predict each query result, execute it, and explain any difference.

`DATABASE_NAME` is initially `:memory:`, so closing the notebook removes the database.
Change it to a filename such as `chapter_database.db` when you want SQLite to create a
persistent database in the notebook's working directory. Do not switch to a persistent
file until the in-memory version runs successfully from top to bottom.


In [1]:
import sqlite3

print(f"Python {__import__('sys').version.split()[0]}; SQLite {sqlite3.sqlite_version}")
DATABASE_NAME = ":memory:"  # Change to "chapter_database.db" to keep a database file.
connection = sqlite3.connect(DATABASE_NAME, isolation_level=None)
connection.execute("PRAGMA foreign_keys = ON")


def run_sql_script(connection, script, max_rows=20):
    """Execute a SQLite script and display result-producing statements."""
    buffer = ""
    for raw_line in script.splitlines():
        stripped = raw_line.strip()
        if stripped.startswith(".print"):
            message = stripped[len(".print"):].strip().strip("\"'")
            print(f"\n{message}")
            continue
        buffer += raw_line + "\n"
        if not sqlite3.complete_statement(buffer):
            continue
        statement = buffer.strip()
        buffer = ""
        if not statement:
            continue
        cursor = connection.execute(statement)
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchmany(max_rows + 1)
            print(" | ".join(columns))
            for row in rows[:max_rows]:
                print(" | ".join("NULL" if value is None else str(value) for value in row))
            if len(rows) > max_rows:
                print(f"... additional rows omitted after {max_rows}")
    remaining = "\n".join(
        line for line in buffer.splitlines() if not line.strip().startswith("--")
    ).strip()
    if remaining:
        raise ValueError("The embedded SQL ends with an incomplete statement.")


def inspect_database(connection):
    """Display tables, columns, primary keys, foreign keys, and integrity status."""
    tables = [
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        )
    ]
    print("Tables:", ", ".join(tables) if tables else "none")
    for table in tables:
        columns = connection.execute(f'PRAGMA table_info("{table}")').fetchall()
        primary_key = [row[1] for row in sorted(columns, key=lambda row: row[5]) if row[5]]
        print(f"\n{table}")
        print("  columns:", ", ".join(f"{row[1]} {row[2]}" for row in columns))
        print("  primary key:", ", ".join(primary_key) if primary_key else "none")
        for index_row in connection.execute(f'PRAGMA index_list("{table}")').fetchall():
            if index_row[2] and index_row[3] == "u":
                unique_columns = [
                    row[2]
                    for row in connection.execute(
                        f'PRAGMA index_info("{index_row[1]}")'
                    ).fetchall()
                ]
                print("  unique constraint:", ", ".join(unique_columns))
        foreign_keys = connection.execute(f'PRAGMA foreign_key_list("{table}")').fetchall()
        for foreign_key in foreign_keys:
            print(f"  foreign key: {foreign_key[3]} -> {foreign_key[2]}.{foreign_key[4]}")
    violations = connection.execute("PRAGMA foreign_key_check").fetchall()
    print("\nForeign-key check:", "PASS" if not violations else violations)


Python 3.12.9; SQLite 3.45.3


### Course-registration setup


In [2]:
SQL_1 = """PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS enrollment;
DROP TABLE IF EXISTS course;
DROP TABLE IF EXISTS student;
DROP TABLE IF EXISTS department;

CREATE TABLE department (
    dept_code TEXT PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE,
    building TEXT NOT NULL
);

CREATE TABLE student (
    student_id TEXT PRIMARY KEY,
    email TEXT NOT NULL UNIQUE,
    student_name TEXT NOT NULL,
    dept_code TEXT NOT NULL,
    FOREIGN KEY (dept_code) REFERENCES department (dept_code)
);

CREATE TABLE course (
    course_id TEXT PRIMARY KEY,
    title TEXT NOT NULL,
    dept_code TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 6),
    FOREIGN KEY (dept_code) REFERENCES department (dept_code)
);

CREATE TABLE enrollment (
    student_id TEXT NOT NULL,
    course_id TEXT NOT NULL,
    term TEXT NOT NULL,
    grade TEXT,
    PRIMARY KEY (student_id, course_id, term),
    FOREIGN KEY (student_id) REFERENCES student (student_id),
    FOREIGN KEY (course_id) REFERENCES course (course_id)
);

INSERT INTO department (dept_code, dept_name, building) VALUES
    ('DES', 'Digital Design', 'Hong Hall'),
    ('FIN', 'Finance', 'Cheng Hall'),
    ('IM', 'Information Management', 'Hong Hall');

INSERT INTO student (student_id, email, student_name, dept_code) VALUES
    ('S101', 'an.chen@example.edu', 'An Chen', 'IM'),
    ('S102', 'bea.lin@example.edu', 'Bea Lin', 'FIN'),
    ('S103', 'kai.wu@example.edu', 'Kai Wu', 'IM'),
    ('S104', 'mira.ho@example.edu', 'Mira Ho', 'DES');

INSERT INTO course (course_id, title, dept_code, credits) VALUES
    ('DB201', 'Database Management', 'IM', 3),
    ('FT210', 'Financial Technology', 'FIN', 3),
    ('ML230', 'Machine Learning', 'IM', 3),
    ('WD120', 'Web Design', 'DES', 2);

INSERT INTO enrollment (student_id, course_id, term, grade) VALUES
    ('S101', 'DB201', '115-1', 'A'),
    ('S101', 'FT210', '115-1', 'B+'),
    ('S102', 'FT210', '115-1', 'A-'),
    ('S103', 'DB201', '115-1', 'B'),
    ('S103', 'ML230', '115-1', 'A'),
    ('S104', 'WD120', '115-1', 'A-');
"""


In [3]:
run_sql_script(connection, SQL_1)


### Inspect the Database You Created

Read the output as a schema check: confirm the table names, column types, primary-key order, foreign-key direction, and integrity result.


In [4]:
inspect_database(connection)


Tables: course, department, enrollment, student

course
  columns: course_id TEXT, title TEXT, dept_code TEXT, credits INTEGER
  primary key: course_id
  foreign key: dept_code -> department.dept_code

department
  columns: dept_code TEXT, dept_name TEXT, building TEXT
  primary key: dept_code
  unique constraint: dept_name

enrollment
  columns: student_id TEXT, course_id TEXT, term TEXT, grade TEXT
  primary key: student_id, course_id, term
  foreign key: course_id -> course.course_id
  foreign key: student_id -> student.student_id

student
  columns: student_id TEXT, email TEXT, student_name TEXT, dept_code TEXT
  primary key: student_id
  unique constraint: email
  foreign key: dept_code -> department.dept_code

Foreign-key check: PASS


### Intermediate SQL lab


In [5]:
SQL_2 = """-- Chapter 4: Intermediate SQL
-- Verified with SQLite 3.45.3. Run Chapter 2 course_registration_setup.sql first.
-- Predict the attributes, row count, NULLs, and database changes before each block.

PRAGMA foreign_keys = ON;

-- Example 1: explicit inner joins. Six enrollment rows should be returned.
SELECT s.student_id, s.student_name, c.course_id, c.title, e.grade
FROM student AS s
JOIN enrollment AS e ON e.student_id = s.student_id
JOIN course AS c ON c.course_id = e.course_id
ORDER BY s.student_id, c.course_id;

-- Example 2: NATURAL JOIN silently uses every shared column name. The intermediate
-- result shares both course_id and dept_code with course, so the cross-department
-- S101/FT210 enrollment disappears. The explicit query above returns six rows;
-- this query returns five.
SELECT student_id, student_name, course_id, title
FROM student
NATURAL JOIN enrollment
NATURAL JOIN course
ORDER BY student_id, course_id;

-- Example 3: USING names the intended common column explicitly.
SELECT c.course_id, c.title, d.dept_name
FROM course AS c
JOIN department AS d USING (dept_code)
ORDER BY c.course_id;

-- Example 4: left outer join retains a course that has no enrollment.
SAVEPOINT unmatched_course_demo;

INSERT INTO course (course_id, title, dept_code, credits)
VALUES ('IS250', 'Information Security', 'IM', 3);

SELECT c.course_id, c.title, COUNT(e.student_id) AS enrollment_count
FROM course AS c
LEFT OUTER JOIN enrollment AS e ON e.course_id = c.course_id
GROUP BY c.course_id, c.title
ORDER BY c.course_id;

-- Example 5: a condition in ON preserves every course; the same condition in WHERE
-- removes courses without a matching A/A- enrollment.
SELECT c.course_id, e.student_id, e.grade
FROM course AS c
LEFT OUTER JOIN enrollment AS e
  ON e.course_id = c.course_id
 AND e.grade IN ('A', 'A-')
ORDER BY c.course_id, e.student_id;

SELECT c.course_id, e.student_id, e.grade
FROM course AS c
LEFT OUTER JOIN enrollment AS e ON e.course_id = c.course_id
WHERE e.grade IN ('A', 'A-')
ORDER BY c.course_id, e.student_id;

ROLLBACK TO unmatched_course_demo;
RELEASE unmatched_course_demo;

-- Example 6: right and full outer joins. SQLite 3.39 or later is required.
DROP TABLE IF EXISTS temp.planned_enrollment;
DROP TABLE IF EXISTS temp.planned_student;

CREATE TEMP TABLE planned_student (
    student_id TEXT PRIMARY KEY
);

CREATE TEMP TABLE planned_enrollment (
    student_id TEXT NOT NULL,
    course_id TEXT NOT NULL
);

INSERT INTO planned_student VALUES ('S101'), ('S105');
INSERT INTO planned_enrollment VALUES ('S101', 'DB201'), ('S999', 'AI999');

SELECT ps.student_id AS known_student,
       pe.student_id AS planned_student,
       pe.course_id
FROM planned_student AS ps
RIGHT OUTER JOIN planned_enrollment AS pe
  ON pe.student_id = ps.student_id
ORDER BY pe.student_id;

SELECT ps.student_id AS known_student,
       pe.student_id AS planned_student,
       pe.course_id
FROM planned_student AS ps
FULL OUTER JOIN planned_enrollment AS pe
  ON pe.student_id = ps.student_id
ORDER BY COALESCE(ps.student_id, pe.student_id);

-- Example 7: a view stores a query definition, not this query's current rows.
DROP VIEW IF EXISTS course_enrollment_summary;

CREATE VIEW course_enrollment_summary AS
SELECT c.course_id,
       c.title,
       COUNT(e.student_id) AS enrollment_count
FROM course AS c
LEFT OUTER JOIN enrollment AS e ON e.course_id = c.course_id
GROUP BY c.course_id, c.title;

SELECT course_id, title, enrollment_count
FROM course_enrollment_summary
ORDER BY course_id;

SAVEPOINT view_change_demo;
INSERT INTO enrollment (student_id, course_id, term, grade)
VALUES ('S102', 'DB201', '115-1', NULL);

SELECT course_id, enrollment_count
FROM course_enrollment_summary
WHERE course_id = 'DB201';

ROLLBACK TO view_change_demo;
RELEASE view_change_demo;

SELECT course_id, enrollment_count
FROM course_enrollment_summary
WHERE course_id = 'DB201';

-- Do not uncomment in a shared database. SQLite views are read-only unless an
-- INSTEAD OF trigger is supplied; the verifier confirms that this update fails.
-- UPDATE course_enrollment_summary
-- SET enrollment_count = 99
-- WHERE course_id = 'DB201';

-- Example 8: a multi-statement course swap is one transaction-sized task.
-- SAVEPOINT lets the lab show and then undo the complete unit of work.
SAVEPOINT course_swap;

DELETE FROM enrollment
WHERE student_id = 'S101'
  AND course_id = 'FT210'
  AND term = '115-1';

INSERT INTO enrollment (student_id, course_id, term, grade)
VALUES ('S101', 'ML230', '115-1', NULL);

SELECT student_id, course_id, term, grade
FROM enrollment
WHERE student_id = 'S101'
ORDER BY course_id;

ROLLBACK TO course_swap;
RELEASE course_swap;

SELECT student_id, course_id, term, grade
FROM enrollment
WHERE student_id = 'S101'
ORDER BY course_id;

-- Example 9: NOT NULL, UNIQUE, CHECK, foreign keys, a default, and a
-- deliberate ON DELETE CASCADE rule.
DROP TABLE IF EXISTS waitlist_entry;

CREATE TABLE waitlist_entry (
    request_id INTEGER PRIMARY KEY,
    student_id TEXT NOT NULL,
    course_id TEXT NOT NULL,
    term TEXT NOT NULL DEFAULT '115-1',
    priority INTEGER NOT NULL CHECK (priority BETWEEN 1 AND 5),
    UNIQUE (student_id, course_id, term),
    FOREIGN KEY (student_id) REFERENCES student (student_id)
        ON DELETE CASCADE,
    FOREIGN KEY (course_id) REFERENCES course (course_id)
);

INSERT INTO waitlist_entry (request_id, student_id, course_id, priority)
VALUES (1, 'S104', 'ML230', 2);

SELECT request_id, student_id, course_id, term, priority
FROM waitlist_entry;

-- Invalid examples are comments so the complete script continues. The verifier
-- executes each one independently and confirms rejection.
-- Duplicate request: same student_id, course_id, and term.
-- INSERT INTO waitlist_entry VALUES (2, 'S104', 'ML230', '115-1', 3);
-- Invalid range: priority 8 violates CHECK.
-- INSERT INTO waitlist_entry VALUES (3, 'S103', 'FT210', '115-1', 8);
-- Missing parent: S999 violates the student foreign key.
-- INSERT INTO waitlist_entry VALUES (4, 'S999', 'FT210', '115-1', 3);

-- Example 10: the chosen cascade deletes dependent waitlist rows. The whole
-- demonstration is rolled back, so S105 is not left in the base data.
SAVEPOINT cascade_demo;

INSERT INTO student (student_id, email, student_name, dept_code)
VALUES ('S105', 'noah.lee@example.edu', 'Noah Lee', 'IM');

INSERT INTO waitlist_entry
    (request_id, student_id, course_id, term, priority)
VALUES (5, 'S105', 'FT210', '115-1', 3);

DELETE FROM student WHERE student_id = 'S105';

SELECT COUNT(*) AS remaining_s105_waitlist_rows
FROM waitlist_entry
WHERE student_id = 'S105';

ROLLBACK TO cascade_demo;
RELEASE cascade_demo;

-- Example 11: CHECK alone does not reject NULL because UNKNOWN is not FALSE.
DROP TABLE IF EXISTS temp.check_without_not_null;
CREATE TEMP TABLE check_without_not_null (
    value INTEGER CHECK (value > 0)
);
INSERT INTO check_without_not_null VALUES (NULL);
SELECT value IS NULL AS null_was_accepted
FROM check_without_not_null;

-- Student practice. Save predictions and actual results.
-- P1. Use explicit JOIN ... ON to list every enrollment with student email and
--     department name of the course. State why each join condition is needed.
-- P2. Rewrite Example 3 using JOIN ... ON without changing its result.
-- P3. Use a left outer join to list every student and the number of courses taken.
-- P4. Predict which course rows disappear when the A/A- condition moves from ON
--     to WHERE, then verify.
-- P5. Create a view named im_course that exposes only course_id, title, and credits
--     for IM courses; query it for three-credit courses.
-- P6. Inside a savepoint, perform a different valid two-statement course swap and
--     prove that ROLLBACK restores the starting rows.
-- P7. Design course_feedback so rating is 1-5, one row exists per enrollment, and
--     feedback cannot refer to a nonexistent enrollment.
"""


In [6]:
run_sql_script(connection, SQL_2)


student_id | student_name | course_id | title | grade
S101 | An Chen | DB201 | Database Management | A
S101 | An Chen | FT210 | Financial Technology | B+
S102 | Bea Lin | FT210 | Financial Technology | A-
S103 | Kai Wu | DB201 | Database Management | B
S103 | Kai Wu | ML230 | Machine Learning | A
S104 | Mira Ho | WD120 | Web Design | A-
student_id | student_name | course_id | title
S101 | An Chen | DB201 | Database Management
S102 | Bea Lin | FT210 | Financial Technology
S103 | Kai Wu | DB201 | Database Management
S103 | Kai Wu | ML230 | Machine Learning
S104 | Mira Ho | WD120 | Web Design
course_id | title | dept_name
DB201 | Database Management | Information Management
FT210 | Financial Technology | Finance
ML230 | Machine Learning | Information Management
WD120 | Web Design | Digital Design
course_id | title | enrollment_count
DB201 | Database Management | 2
FT210 | Financial Technology | 2
IS250 | Information Security | 0
ML230 | Machine Learning | 1
WD120 | Web Design | 1
course_i

### Reproducibility Check


In [7]:
assert connection.execute("SELECT COUNT(*) FROM department").fetchone()[0] == 3
assert connection.execute("SELECT COUNT(*) FROM student").fetchone()[0] == 4
assert connection.execute("SELECT COUNT(*) FROM course").fetchone()[0] == 4
assert connection.execute("SELECT COUNT(*) FROM enrollment").fetchone()[0] == 6
assert connection.execute("PRAGMA foreign_key_check").fetchall() == []
print("Notebook checks passed.")


Notebook checks passed.


In [8]:
connection.close()
print("In-memory database closed.")


In-memory database closed.
